# Fixed-point solver benchmarks

Compare this project's symbolic `FPJ` implementation with the four explicit Python ports of Oink's `FPI`, `FPIM`, `FPJ`, and `FPJM` solvers. The notebook can benchmark Lily by itself or every bundled paper dataset. It checks that all solvers agree on winning regions before producing per-game results, cumulative runtimes, speedups, iteration and strategy-size summaries, dataset statistics, and BDD-node tables.

Timing runs leave `tracemalloc` disabled because allocation tracing changes runtime. Four worker processes are used by default; set `WORKERS = 1` for sequential execution.

In [ ]:
from pathlib import Path
import gc
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from parity_bdd_lab import (
    benchmark_all_datasets,
    benchmark_dataset,
    benchmark_dataset_names,
    collect_bdd_garbage,
    iter_benchmark_files,
    read_pg,
    records_to_dataframe,
    write_benchmark_csv,
)
from parity_bdd_lab.fpj import _Encoding

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## Configuration

`FPJ` is the project's symbolic implementation. Names beginning with `Oink-` select the explicit ports. Enable either or both benchmark scopes. When both are enabled, the all-dataset result is selected for the tables because it already contains Lily.

In [ ]:
METHODS = ('FPJ', 'Oink-FPI', 'Oink-FPIM', 'Oink-FPJ', 'Oink-FPJM')
REPEATS = 3
WORKERS = 4
RUN_LILY = True
RUN_ALL_DATASETS = False
WRITE_CSV = False

assert RUN_LILY or RUN_ALL_DATASETS, 'Enable at least one benchmark scope'
benchmark_dataset_names()

## Run the Lily benchmark

This cell runs every Lily game for every selected method. Progress lines are emitted as worker processes finish runs.

In [ ]:
if RUN_LILY:
    lily_records = benchmark_dataset(
        'Lily',
        methods=METHODS,
        repeats=REPEATS,
        measure_memory=False,
        workers=WORKERS,
    )
    lily_results = records_to_dataframe(lily_records)
    expected_games = len(tuple(iter_benchmark_files('Lily')))
    assert len(lily_results) == expected_games * len(METHODS) * REPEATS
    display(lily_results)

## Run all datasets

Set `RUN_ALL_DATASETS = True` above to benchmark the complete bundled collection. This can take substantially longer than Lily alone.

In [ ]:
if RUN_ALL_DATASETS:
    all_records = benchmark_all_datasets(
        methods=METHODS,
        repeats=REPEATS,
        measure_memory=False,
        workers=WORKERS,
    )
    all_results = records_to_dataframe(all_records)
    expected_games = sum(
        len(tuple(iter_benchmark_files(dataset)))
        for dataset in benchmark_dataset_names()
    )
    assert len(all_results) == expected_games * len(METHODS) * REPEATS
    display(all_results)

## Select and optionally save the active results

All tables below use `benchmark_results`. Re-running only the analysis cells does not repeat solver runs.

In [ ]:
if RUN_ALL_DATASETS:
    benchmark_records = all_records
    benchmark_results = all_results.copy()
    benchmark_scope = 'all_datasets'
else:
    benchmark_records = lily_records
    benchmark_results = lily_results.copy()
    benchmark_scope = 'lily'

if WRITE_CSV:
    csv_path = write_benchmark_csv(
        benchmark_records,
        CODE / 'data' / 'benchmarks' / f'fixed_point_{benchmark_scope}.csv',
    )
    print(f'Wrote {csv_path}')

print(f'{benchmark_scope}: {benchmark_results.game.nunique()} games, '      f'{len(METHODS)} methods, {REPEATS} repeats')

## Correctness agreement

A game passes when every method and repeat returns the same winning-region fingerprint. A failing row should be investigated before interpreting timings.

In [ ]:
agreement_table = (
    benchmark_results
    .groupby(['dataset', 'game'], as_index=False)
    .agg(
        methods=('method', 'nunique'),
        runs=('solution_fingerprint', 'size'),
        distinct_solutions=('solution_fingerprint', 'nunique'),
    )
)
agreement_table['agrees'] = agreement_table['distinct_solutions'].eq(1)
display(agreement_table)
assert agreement_table['agrees'].all(), agreement_table[~agreement_table['agrees']]

## Per-game benchmark table

Repeated runs are reduced to mean and standard deviation per game and method. The first strategy size, iteration count, and BDD count are retained because they are deterministic solver outputs.

In [ ]:
per_game = (
    benchmark_results
    .groupby(['dataset', 'game', 'method'], as_index=False)
    .agg(
        mean_seconds=('runtime_seconds', 'mean'),
        std_seconds=('runtime_seconds', 'std'),
        min_seconds=('runtime_seconds', 'min'),
        max_seconds=('runtime_seconds', 'max'),
        iterations=('iterations', 'first'),
        strategy_edges=('strategy_edges', 'first'),
        even_winning_vertices=('even_winning_vertices', 'first'),
        odd_winning_vertices=('odd_winning_vertices', 'first'),
        bdd_nodes=('bdd_nodes', 'first'),
    )
)
display(per_game.sort_values(['dataset', 'game', 'method']).reset_index(drop=True))

## Cumulative runtime by dataset

For each repeat, runtime is summed over all games in the dataset; the table reports the mean cumulative runtime across repeats.

In [ ]:
dataset_order = [
    dataset for dataset in benchmark_dataset_names()
    if dataset in set(benchmark_results['dataset'])
]
per_repeat_runtime = (
    benchmark_results
    .groupby(['dataset', 'method', 'run'], as_index=False)['runtime_seconds']
    .sum()
)
cumulative_runtime = (
    per_repeat_runtime
    .groupby(['dataset', 'method'])['runtime_seconds']
    .mean()
    .unstack('method')
    .reindex(index=dataset_order, columns=METHODS)
)
display(cumulative_runtime.round(4))

## Overall method summary and speedup over symbolic FPJ

`total_seconds` sums the per-game mean runtimes, so each game contributes once regardless of `REPEATS`. A speedup above one is faster than the project's symbolic FPJ baseline.

In [ ]:
method_summary = (
    per_game
    .groupby('method', as_index=False)
    .agg(
        datasets=('dataset', 'nunique'),
        games=('game', 'size'),
        total_seconds=('mean_seconds', 'sum'),
        median_game_seconds=('mean_seconds', 'median'),
        max_game_seconds=('mean_seconds', 'max'),
        mean_iterations=('iterations', 'mean'),
        total_strategy_edges=('strategy_edges', 'sum'),
    )
    .set_index('method')
    .reindex(METHODS)
)
fpj_total = method_summary.loc['FPJ', 'total_seconds']
method_summary['speedup_vs_symbolic_fpj'] = fpj_total / method_summary['total_seconds']
display(method_summary.round(4))

## Iterations and returned strategy edges by dataset

FPI and FPJ iteration counters have algorithm-specific meanings, so compare them as implementation work indicators rather than identical operations. FPIM and FPJM return multi-strategies and can retain more edges than their positional counterparts.

In [ ]:
iteration_summary = (
    per_game.groupby(['dataset', 'method'])['iterations'].mean()
    .unstack('method').reindex(index=dataset_order, columns=METHODS)
)
strategy_summary = (
    per_game.groupby(['dataset', 'method'])['strategy_edges'].mean()
    .unstack('method').reindex(index=dataset_order, columns=METHODS)
)
display(iteration_summary.round(1).style.set_caption('Mean iterations per game'))
display(strategy_summary.round(1).style.set_caption('Mean returned strategy edges per game'))

## Dataset graph and static BDD statistics

This reproduces the structural dataset table used by the other benchmark notebook. The static BDD count measures the symbolic encoding before solving; it is independent of the explicit Oink ports.

In [ ]:
game_statistics = []
for dataset in dataset_order:
    for game_path in iter_benchmark_files(dataset):
        game = read_pg(game_path)
        priorities = tuple(game.priority[vertex] for vertex in game.vertices)
        encoding = _Encoding(game)
        collect_bdd_garbage(encoding.bdd)
        game_statistics.append({
            'dataset': dataset,
            'game': game_path.name,
            'vertices': len(game.vertices),
            'edges': sum(len(game.successors[v]) for v in game.vertices),
            'min_priority': min(priorities),
            'max_priority': max(priorities),
            'static_bdd_nodes': len(encoding.bdd),
        })
        del encoding, game
        gc.collect()

game_statistics = pd.DataFrame(game_statistics)
dataset_statistics = (
    game_statistics.groupby('dataset')
    .apply(lambda group: pd.Series({
        'games': group['game'].nunique(),
        'mean_vertices': group['vertices'].mean(),
        'max_vertices': group['vertices'].max(),
        'priority_range': f"{group['min_priority'].min()}–{group['max_priority'].max()}",
        'mean_out_degree': group['edges'].sum() / group['vertices'].sum(),
        'mean_static_bdd_nodes': group['static_bdd_nodes'].mean(),
    }), include_groups=False)
    .reindex(dataset_order)
)
display(dataset_statistics.round({
    'mean_vertices': 0, 'mean_out_degree': 2, 'mean_static_bdd_nodes': 0
}))

## Returned BDD nodes by dataset

Only the symbolic FPJ implementation reports a BDD-manager size. Explicit Oink solvers therefore appear blank.

In [ ]:
bdd_node_summary = (
    benchmark_results.groupby(['dataset', 'method'])['bdd_nodes'].mean()
    .unstack('method').reindex(index=dataset_order, columns=METHODS)
)
display(bdd_node_summary.round(0))

## Runtime overview

The logarithmic scale keeps both fast explicit runs and slower symbolic runs visible.

In [ ]:
plot_data = per_game.pivot_table(
    index=['dataset', 'game'], columns='method', values='mean_seconds'
).reindex(columns=METHODS)
ax = plot_data.plot(figsize=(14, 6), marker='.', linewidth=1)
ax.set_yscale('log')
ax.set_xlabel('Game (grouped by dataset)')
ax.set_ylabel('Mean runtime (seconds, log scale)')
ax.set_title(f'Fixed-point solver runtimes: {benchmark_scope}')
ax.grid(axis='y', which='both', alpha=0.25)
plt.tight_layout()
plt.show()